
# Canadian Household Spending Analysis — Food & Consumer Comparative Trends
**Analyst:** Carlos Restrepo | GLOCAL Foundation of Canada  
**Phase:** Comparative Sector Analysis — Food & Consumer  
**Source:** Statistics Canada — SHS PUMF 2017, 2019, 2021  
**Kernel:** Python (shs2021)

## Business Objective
This notebook compares **2017, 2019, and 2021** to show how Canadian household food spending evolved across the pre-COVID and COVID-affected period. The focus is on **trend lines and clear comparative insights** that matter for grocery retailers, CPG companies, food delivery platforms, and restaurant chains.

## Core Comparative Questions
1. How did **total food spending** evolve from 2017 to 2021?
2. How did the split between **food from stores vs. restaurants** change across the three years?
3. Which **food subcategories** gained or lost relative importance?
4. How did the food mix shift by **income quintile** and **province**?
5. What do the three years suggest about **consumer pressure, value sensitivity, and channel shifts**?


In [ ]:

# ============================================================
# Cell 2 - Imports, paths, and helper functions
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
import os

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)

base = Path(r"C:\Users\LENOVO\canadian_household_spending")
output_dir = base / 'outputs' / 'comparative' / 'food'
os.makedirs(output_dir, exist_ok=True)

YEARS = [2017, 2019, 2021]

paths = {
    2017: base / 'outputs' / '2017' / 'shs_2017_clean.csv',
    2019: base / 'outputs' / '2019' / 'shs_2019_clean.csv',
    2021: base / 'outputs' / '2021' / 'shs_2021_clean.csv',
}

def ensure_column(df, candidates, target_name):
    for c in candidates:
        if c in df.columns:
            if c != target_name:
                df = df.rename(columns={c: target_name})
            return df
    raise KeyError(f"Could not find any of these columns for {target_name}: {candidates}")


def standardize_columns(df):
    df = ensure_column(df, ['WeightD', 'WEIGHTD', 'weightd'], 'WeightD')
    df = ensure_column(df, ['HH_TOTAL_INCOME', 'HH_TotInc', 'HHTOTINC', 'HH_TOTINC', 'hh_totinc'], 'HH_TOTAL_INCOME')
    df = ensure_column(df, ['INCOME_QUINTILE', 'income_quintile'], 'INCOME_QUINTILE')
    df = ensure_column(df, ['PROV_NAME', 'prov_name'], 'PROV_NAME')

    rename_pairs = {
        'fd001': 'FD001', 'fd003': 'FD003', 'fd990': 'FD990',
        'fd100': 'FD100', 'fd200': 'FD200', 'fd300': 'FD300', 'fd400': 'FD400',
        'fd500': 'FD500', 'fd600': 'FD600', 'fd700': 'FD700', 'fd800': 'FD800',
    }
    for old, new in rename_pairs.items():
        if old in df.columns and new not in df.columns:
            df = df.rename(columns={old: new})
    return df


def weighted_mean(df, var, weight='WeightD'):
    valid = df[[var, weight]].dropna()
    if valid.empty or valid[weight].sum() == 0:
        return np.nan
    return (valid[var] * valid[weight]).sum() / valid[weight].sum()


def pct_change(first, last):
    if pd.isna(first) or first == 0:
        return np.nan
    return (last / first - 1) * 100


food_store_subcats = {
    'FD100': 'Bakery products',
    'FD200': 'Cereal grains & products',
    'FD300': 'Fruit, preparations & nuts',
    'FD400': 'Vegetables & preparations',
    'FD500': 'Dairy products & eggs',
    'FD600': 'Meat',
    'FD700': 'Fish & seafood',
    'FD800': 'Non-alcoholic beverages & other',
}

quintile_order = ['Q1 - Lowest', 'Q2', 'Q3', 'Q4', 'Q5 - Highest']


In [ ]:

# ============================================================
# Cell 3 - Load and harmonize yearly datasets
# ============================================================
dfs = {}
for year, path in paths.items():
    df = pd.read_csv(path)
    df = standardize_columns(df)
    df['Year'] = year
    dfs[year] = df.copy()
    print(f"{year}: rows={df.shape[0]:,}, cols={df.shape[1]:,}")

print("\nCommon analytic columns verified.")



## National trend overview
These tables and charts show the big-picture movement in household food spending, including the shift between **stores and restaurants**.


In [ ]:

# ============================================================
# Cell 4 - National trend table
# ============================================================
national_rows = []
for year in YEARS:
    d = dfs[year]
    total_food = weighted_mean(d, 'FD001')
    stores = weighted_mean(d, 'FD003')
    restaurants = weighted_mean(d, 'FD990')
    income = weighted_mean(d, 'HH_TOTAL_INCOME')
    national_rows.append({
        'Year': year,
        'Avg Income': round(income, 1),
        'Avg Food': round(total_food, 1),
        'Food from Stores': round(stores, 1),
        'Food from Restaurants': round(restaurants, 1),
        'Stores Share %': round(stores / total_food * 100, 1),
        'Restaurant Share %': round(restaurants / total_food * 100, 1),
        'Food Burden %': round(total_food / income * 100, 1),
    })

national_df = pd.DataFrame(national_rows)
display(national_df.style.hide(axis='index').format({
    'Avg Income': '${:,.1f}',
    'Avg Food': '${:,.1f}',
    'Food from Stores': '${:,.1f}',
    'Food from Restaurants': '${:,.1f}',
    'Stores Share %': '{:.1f}%',
    'Restaurant Share %': '{:.1f}%',
    'Food Burden %': '{:.1f}%',
}))

national_df.to_csv(output_dir / 'food_national_trends.csv', index=False)


In [ ]:

# ============================================================
# Cell 5 - National trend charts
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Total food trend
ax = axes[0, 0]
ax.plot(national_df['Year'], national_df['Avg Food'], marker='o', linewidth=2.5)
for x, y in zip(national_df['Year'], national_df['Avg Food']):
    ax.annotate(f'${y:,.0f}', (x, y), textcoords='offset points', xytext=(0,8), ha='center')
ax.set_title('Average Household Food Spending', fontweight='bold')
ax.set_ylabel('CAD ($)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.set_xticks(YEARS)

# Stores vs restaurants share
ax = axes[0, 1]
ax.plot(national_df['Year'], national_df['Stores Share %'], marker='o', linewidth=2.5, label='Stores share')
ax.plot(national_df['Year'], national_df['Restaurant Share %'], marker='o', linewidth=2.5, label='Restaurant share')
for series in ['Stores Share %', 'Restaurant Share %']:
    for x, y in zip(national_df['Year'], national_df[series]):
        ax.annotate(f'{y:.1f}%', (x, y), textcoords='offset points', xytext=(0,8), ha='center')
ax.set_title('Channel Mix Trend', fontweight='bold')
ax.set_ylabel('% of total food spending')
ax.set_xticks(YEARS)
ax.legend(frameon=False)

# Dollar split
ax = axes[1, 0]
ax.plot(national_df['Year'], national_df['Food from Stores'], marker='o', linewidth=2.5, label='Stores')
ax.plot(national_df['Year'], national_df['Food from Restaurants'], marker='o', linewidth=2.5, label='Restaurants')
for series in ['Food from Stores', 'Food from Restaurants']:
    for x, y in zip(national_df['Year'], national_df[series]):
        ax.annotate(f'${y:,.0f}', (x, y), textcoords='offset points', xytext=(0,8), ha='center')
ax.set_title('Food Spending by Channel', fontweight='bold')
ax.set_ylabel('CAD ($)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.set_xticks(YEARS)
ax.legend(frameon=False)

# Food burden trend
ax = axes[1, 1]
ax.plot(national_df['Year'], national_df['Food Burden %'], marker='o', linewidth=2.5)
for x, y in zip(national_df['Year'], national_df['Food Burden %']):
    ax.annotate(f'{y:.1f}%', (x, y), textcoords='offset points', xytext=(0,8), ha='center')
ax.set_title('Food Burden Trend', fontweight='bold')
ax.set_ylabel('% of household income')
ax.set_xticks(YEARS)

plt.tight_layout()
plt.savefig(output_dir / 'food_comparative_national_trends.png', dpi=150, bbox_inches='tight')
plt.show()



## Food subcategory trends
This section tracks how the **main food-from-stores subcategories** changed over time in both **dollar terms** and **share of total food**.


In [ ]:

# ============================================================
# Cell 6 - Food subcategory comparative table
# ============================================================
subcat_rows = []
for year in YEARS:
    d = dfs[year]
    total_food = weighted_mean(d, 'FD001')
    for code, label in food_store_subcats.items():
        value = weighted_mean(d, code)
        subcat_rows.append({
            'Year': year,
            'Subcategory': label,
            'Spending': round(value, 1),
            'Share of Total Food %': round(value / total_food * 100, 1)
        })

subcat_df = pd.DataFrame(subcat_rows)
display(subcat_df.pivot(index='Subcategory', columns='Year', values='Spending').round(1))
subcat_df.to_csv(output_dir / 'food_subcategory_trends.csv', index=False)


In [ ]:

# ============================================================
# Cell 7 - Food subcategory trend charts
# ============================================================
spend_pivot = subcat_df.pivot(index='Subcategory', columns='Year', values='Spending').loc[list(food_store_subcats.values())]
share_pivot = subcat_df.pivot(index='Subcategory', columns='Year', values='Share of Total Food %').loc[list(food_store_subcats.values())]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for subcat in spend_pivot.index:
    axes[0].plot(YEARS, spend_pivot.loc[subcat, YEARS], marker='o', linewidth=2, label=subcat)
axes[0].set_title('Food Subcategory Spending Trends', fontweight='bold')
axes[0].set_ylabel('CAD ($)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[0].set_xticks(YEARS)

for subcat in share_pivot.index:
    axes[1].plot(YEARS, share_pivot.loc[subcat, YEARS], marker='o', linewidth=2, label=subcat)
axes[1].set_title('Food Subcategory Share of Total Food', fontweight='bold')
axes[1].set_ylabel('% of total food spending')
axes[1].set_xticks(YEARS)

handles, labels = axes[1].get_legend_handles_labels()
fig.legend(handles, labels, loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False)
plt.tight_layout(rect=[0, 0, 0.82, 1])
plt.savefig(output_dir / 'food_comparative_subcategory_trends.png', dpi=150, bbox_inches='tight')
plt.show()



## Income segmentation trends
These views show which income groups drove change, and whether the market shifted more toward **value-oriented grocery behavior** or a stronger **restaurant mix**.


In [ ]:

# ============================================================
# Cell 8 - Quintile comparison table
# ============================================================
q_rows = []
for year in YEARS:
    d = dfs[year]
    for q in quintile_order:
        subset = d[d['INCOME_QUINTILE'] == q]
        total_food = weighted_mean(subset, 'FD001')
        income = weighted_mean(subset, 'HH_TOTAL_INCOME')
        stores = weighted_mean(subset, 'FD003')
        restaurants = weighted_mean(subset, 'FD990')
        q_rows.append({
            'Year': year,
            'Quintile': q,
            'Avg Income': round(income, 1),
            'Total Food': round(total_food, 1),
            'Stores': round(stores, 1),
            'Restaurants': round(restaurants, 1),
            'Food Burden %': round(total_food / income * 100, 1),
            'Restaurant Share %': round(restaurants / total_food * 100, 1),
        })

quintile_df = pd.DataFrame(q_rows)
display(quintile_df.style.hide(axis='index').format({
    'Avg Income': '${:,.1f}',
    'Total Food': '${:,.1f}',
    'Stores': '${:,.1f}',
    'Restaurants': '${:,.1f}',
    'Food Burden %': '{:.1f}%',
    'Restaurant Share %': '{:.1f}%',
}))

quintile_df.to_csv(output_dir / 'food_quintile_trends.csv', index=False)


In [ ]:

# ============================================================
# Cell 9 - Quintile trend charts
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for q in quintile_order:
    series = quintile_df[quintile_df['Quintile'] == q].sort_values('Year')
    axes[0].plot(series['Year'], series['Food Burden %'], marker='o', linewidth=2, label=q)
axes[0].set_title('Food Burden Trend by Income Quintile', fontweight='bold')
axes[0].set_ylabel('% of household income')
axes[0].set_xticks(YEARS)

for q in quintile_order:
    series = quintile_df[quintile_df['Quintile'] == q].sort_values('Year')
    axes[1].plot(series['Year'], series['Restaurant Share %'], marker='o', linewidth=2, label=q)
axes[1].set_title('Restaurant Share Trend by Income Quintile', fontweight='bold')
axes[1].set_ylabel('% of total food spending')
axes[1].set_xticks(YEARS)

handles, labels = axes[1].get_legend_handles_labels()
fig.legend(handles, labels, loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False)
plt.tight_layout(rect=[0, 0, 0.82, 1])
plt.savefig(output_dir / 'food_comparative_quintile_trends.png', dpi=150, bbox_inches='tight')
plt.show()



## Provincial comparison
This section identifies which provinces became more food-burdened, and where the restaurant channel held up better or weakened more sharply.


In [ ]:

# ============================================================
# Cell 10 - Provincial comparison table
# ============================================================
prov_rows = []
for year in YEARS:
    d = dfs[year]
    for prov in sorted(d['PROV_NAME'].dropna().unique()):
        subset = d[d['PROV_NAME'] == prov]
        total_food = weighted_mean(subset, 'FD001')
        income = weighted_mean(subset, 'HH_TOTAL_INCOME')
        restaurants = weighted_mean(subset, 'FD990')
        prov_rows.append({
            'Year': year,
            'Province': prov,
            'Avg Income': round(income, 1),
            'Total Food': round(total_food, 1),
            'Food Burden %': round(total_food / income * 100, 1),
            'Restaurant Share %': round(restaurants / total_food * 100, 1),
        })

prov_df = pd.DataFrame(prov_rows)
display(prov_df.head(15).style.hide(axis='index').format({
    'Avg Income': '${:,.1f}',
    'Total Food': '${:,.1f}',
    'Food Burden %': '{:.1f}%',
    'Restaurant Share %': '{:.1f}%',
}))

prov_df.to_csv(output_dir / 'food_province_trends.csv', index=False)


In [ ]:

# ============================================================
# Cell 11 - Provincial trend charts
# ============================================================
latest_prov = prov_df[prov_df['Year'] == 2021].sort_values('Food Burden %', ascending=False)['Province'].head(6).tolist()
rest_top_prov = prov_df[prov_df['Year'] == 2021].sort_values('Restaurant Share %', ascending=False)['Province'].head(6).tolist()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for prov in latest_prov:
    series = prov_df[prov_df['Province'] == prov].sort_values('Year')
    axes[0].plot(series['Year'], series['Food Burden %'], marker='o', linewidth=2, label=prov)
axes[0].set_title('Food Burden Trend — Highest-Burden Provinces in 2021', fontweight='bold')
axes[0].set_ylabel('% of household income')
axes[0].set_xticks(YEARS)

for prov in rest_top_prov:
    series = prov_df[prov_df['Province'] == prov].sort_values('Year')
    axes[1].plot(series['Year'], series['Restaurant Share %'], marker='o', linewidth=2, label=prov)
axes[1].set_title('Restaurant Share Trend — Top Provinces in 2021', fontweight='bold')
axes[1].set_ylabel('% of total food spending')
axes[1].set_xticks(YEARS)

handles, labels = axes[1].get_legend_handles_labels()
fig.legend(handles, labels, loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False)
plt.tight_layout(rect=[0, 0, 0.82, 1])
plt.savefig(output_dir / 'food_comparative_provincial_trends.png', dpi=150, bbox_inches='tight')
plt.show()



## Executive comparative findings
The next cell summarizes the biggest shifts across the three years in language that is ready to use for sector commentary.


In [ ]:

# ============================================================
# Cell 12 - Executive comparative summary
# ============================================================
row_2017 = national_df[national_df['Year'] == 2017].iloc[0]
row_2019 = national_df[national_df['Year'] == 2019].iloc[0]
row_2021 = national_df[national_df['Year'] == 2021].iloc[0]

q1_2017 = quintile_df[(quintile_df['Year'] == 2017) & (quintile_df['Quintile'] == 'Q1 - Lowest')].iloc[0]
q1_2021 = quintile_df[(quintile_df['Year'] == 2021) & (quintile_df['Quintile'] == 'Q1 - Lowest')].iloc[0]
q5_2017 = quintile_df[(quintile_df['Year'] == 2017) & (quintile_df['Quintile'] == 'Q5 - Highest')].iloc[0]
q5_2021 = quintile_df[(quintile_df['Year'] == 2021) & (quintile_df['Quintile'] == 'Q5 - Highest')].iloc[0]

subcat_change = subcat_df.pivot(index='Subcategory', columns='Year', values='Spending')
subcat_change['Change_2017_2021_%'] = ((subcat_change[2021] / subcat_change[2017]) - 1) * 100
fastest_growing = subcat_change['Change_2017_2021_%'].sort_values(ascending=False).index[0]
slowest_growing = subcat_change['Change_2017_2021_%'].sort_values(ascending=True).index[0]

prov_change = prov_df.pivot(index='Province', columns='Year', values='Food Burden %')
prov_change['Change_2017_2021_pp'] = prov_change[2021] - prov_change[2017]
most_pressure = prov_change['Change_2017_2021_pp'].sort_values(ascending=False).index[0]
least_pressure = prov_change['Change_2017_2021_pp'].sort_values(ascending=True).index[0]

print("=== FOOD & CONSUMER COMPARATIVE FINDINGS: 2017 vs 2019 vs 2021 ===\n")
print(f"1. Total food spending moved from ${row_2017['Avg Food']:,.1f} in 2017 to ${row_2021['Avg Food']:,.1f} in 2021 ({pct_change(row_2017['Avg Food'], row_2021['Avg Food']):.1f}%).")
print(f"2. Store share rose from {row_2017['Stores Share %']:.1f}% to {row_2021['Stores Share %']:.1f}%, while restaurant share fell from {row_2017['Restaurant Share %']:.1f}% to {row_2021['Restaurant Share %']:.1f}%.")
print(f"3. The lowest-income quintile stayed under the greatest pressure: food burden moved from {q1_2017['Food Burden %']:.1f}% in 2017 to {q1_2021['Food Burden %']:.1f}% in 2021, versus {q5_2017['Food Burden %']:.1f}% to {q5_2021['Food Burden %']:.1f}% for the highest quintile.")
print(f"4. The fastest-growing store subcategory between 2017 and 2021 was {fastest_growing}, while the weakest relative growth came from {slowest_growing}.")
print(f"5. Province-level pressure increased the most in {most_pressure} and eased the most in {least_pressure} over the 2017–2021 period.")
print("\nBusiness reading:")
print("- 2017 works as a pre-shift baseline.")
print("- 2019 still shows a stronger restaurant mix than 2021.")
print("- 2021 is more home-centric, more grocery-led, and more useful for reading value and convenience demand inside the household.")
